In [ ]:
import jax
import jax.numpy as jnp

# Create our mesh! We're running on a TPU v2-8 4x2 slice with names 'X' and 'Y'.
# The Auto axis type tells JAX to let the XLA compiler infer intermediate shardings.
assert len(jax.devices()) == 8
Auto = jax.sharding.AxisType.Auto
mesh = jax.make_mesh(axis_shapes=(4, 2), axis_names=('X', 'Y'), axis_types=(Auto, Auto))

# A little utility function to help define our sharding. A PartitionSpec is our
# sharding (a mapping from axes to names).
def P(*args):
  return jax.NamedSharding(mesh, jax.sharding.PartitionSpec(*args))

A = jnp.zeros((8192, 8192), dtype=jnp.bfloat16, device=P('X', None)) # AllGather
B = jnp.zeros((8192, 8192), dtype=jnp.bfloat16, device=P(None, None)) # ReduceScatter
C = jnp.zeros((8192, 8192), dtype=jnp.bfloat16, device=P(None, None)) # AllReduce
D = jnp.zeros((8192, 8192), dtype=jnp.bfloat16, device=P('X', None)) # AllToAll

%timeit -r 5 -n 1 jax.lax.all_gather(A, 'X',)
%timeit -r 5 -n 1 jax.lax.psum_scatter(A, 'Y',)
%timeit -r 5 -n 1 jax.lax.psum(A, 'Y',)
%timeit -r 5 -n 1 jax.lax.all_to_all(A, 'X', 0, 1)

